# 02 Preprocessing and Feature Engineering

This notebook cleans the combined airport security dataset, creates the target variable and engineered features, and saves a processed dataset for modeling.

In [103]:
import pandas as pd
import numpy as np

Loading the data the same was as in the 01_data_understanding notebook.

In [104]:
file_path = "../data/raw/security_checkpoint_data.xlsx"
all_sheets = pd.read_excel(file_path, sheet_name=None)

dfs = []
for sheet_name, sheet_df in all_sheets.items():
    sheet_df["source_sheet"] = sheet_name
    dfs.append(sheet_df)

df = pd.concat(dfs, ignore_index=True)

print(df.shape)
df.head()

(2405, 26)


,Comments,Lane number,Date,business,senior,family,young,PRM,regular,Experience,...,Time Start WTMD check,Time End WTMD check,Time Start ETD check,Time End ETD check,Time Start Baggage reclaim,Time End Baggage reclaim,Group size,Time Start Baggage Check,Time End Baggage Check,source_sheet
0,"2x wtmd, shoes",3,2018-02-23 00:00:00,NaN,NaN,NaN,NaN,NaN,1.0,0.0,...,NaN,NaN,NaN,NaN,50552.0,50732.0,2.0,NaN,NaN,2018_02_23
1,NaN,3,2018-02-23 00:00:00,NaN,NaN,NaN,NaN,NaN,1.0,0.0,...,NaN,NaN,50601.0,50608.0,50610.0,50753.0,2.0,NaN,NaN,2018_02_23
2,NaN,3,2018-02-23 00:00:00,NaN,NaN,NaN,NaN,NaN,1.0,0.0,...,NaN,NaN,NaN,NaN,50735.0,50814.0,2.0,NaN,NaN,2018_02_23
3,2x wtmd,3,2018-02-23 00:00:00,NaN,NaN,NaN,NaN,NaN,1.0,0.0,...,NaN,NaN,50741.0,50748.0,50758.0,50859.0,2.0,NaN,NaN,2018_02_23
4,NaN,3,2018-02-23 00:00:00,1.0,NaN,NaN,NaN,NaN,1.0,1.0,...,NaN,NaN,NaN,NaN,51115.0,51213.0,1.0,NaN,NaN,2018_02_23


In [105]:
df.columns = df.columns.str.strip()
df.columns.tolist()

['Comments',
 'Lane number',
 'Date',
 'business',
 'senior',
 'family',
 'young',
 'PRM',
 'regular',
 'Experience',
 'Time Start Baggage drop off',
 'Number of boxes',
 'Time End Baggage drop off',
 'Time Start WTMD',
 'Time Start WTMD 2',
 'Time Start WTMD 3',
 'Time Start WTMD check',
 'Time End WTMD check',
 'Time Start ETD check',
 'Time End ETD check',
 'Time Start Baggage reclaim',
 'Time End Baggage reclaim',
 'Group size',
 'Time Start Baggage Check',
 'Time End Baggage Check',
 'source_sheet']

In [106]:
passenger_type_cols = ["business", "senior", "family", "young", "PRM", "regular"]

df[passenger_type_cols] = df[passenger_type_cols].fillna(0)
df[passenger_type_cols] = df[passenger_type_cols].astype(int)

df[passenger_type_cols].head()

,business,senior,family,young,PRM,regular
0,0,0,0,0,0,1
1,0,0,0,0,0,1
2,0,0,0,0,0,1
3,0,0,0,0,0,1
4,1,0,0,0,0,1


In [107]:
df["Experience"].value_counts(dropna=False).sort_index()

Experience
0.0     939
1.0    1464
NaN       2
Name: count, dtype: int64

In [108]:
df["Experience"] = df["Experience"].fillna(df["Experience"].mode()[0]).astype(int)

In [109]:
df["Number of boxes"] = df["Number of boxes"].fillna(df["Number of boxes"].median())
df["Group size"] = df["Group size"].fillna(df["Group size"].median())

df["Number of boxes"] = df["Number of boxes"].astype(float)
df["Group size"] = df["Group size"].astype(float)

In [110]:
def hhmmss_to_seconds(value):
    if pd.isna(value):
        return np.nan

    try:
        # First convert numeric-looking values safely
        numeric_value = int(float(value))
    except (ValueError, TypeError):
        return np.nan

    value_str = str(numeric_value).zfill(6)

    hours = int(value_str[:2])
    minutes = int(value_str[2:4])
    seconds = int(value_str[4:6])

    if hours > 23 or minutes > 59 or seconds > 59:
        return np.nan

    return hours * 3600 + minutes * 60 + seconds

In [111]:
time_cols = [
    "Time Start Baggage drop off",
    "Time End Baggage drop off",
    "Time Start WTMD",
    "Time Start WTMD 2",
    "Time Start WTMD 3",
    "Time Start WTMD check",
    "Time End WTMD check",
    "Time Start ETD check",
    "Time End ETD check",
    "Time Start Baggage reclaim",
    "Time End Baggage reclaim",
    "Time Start Baggage Check",
    "Time End Baggage Check"
]

for col in time_cols:
    df[col + "_sec"] = df[col].apply(hhmmss_to_seconds)

In [112]:
for col in time_cols:
    bad_values = df[
        df[col].notna() &
        ~df[col].astype(str).str.strip().str.isdigit()
    ][col].unique()

    if len(bad_values) > 0:
        print(f"\nColumn: {col}")
        print(bad_values[:10])


Column: Time Start Baggage drop off
[50408. 50410. 50654. 51008. 51009. 51010. 51023. 51100. 51224. 51234.]

Column: Time End Baggage drop off
[50526. 50538. 50719. 50723. 51054. 51047. 51056. 51115. 51107. 51125.]

Column: Time Start WTMD
[50535. 50555. 50721. 50727. 51057. 51052. 51109. 51118. 51138. 51142.]

Column: Time Start WTMD 2
[50536. 50728. 53130. 54151. 54320. 55251. 55724. 60106. 60055. 60653.]

Column: Time Start WTMD 3
[143406.  62620.  65256.]

Column: Time Start WTMD check
[51725. 52420. 53201. 53240. 54152. 54950. 60107. 60101. 60000. 60723.]

Column: Time End WTMD check
[51752. 52445. 53205. 53251. 54211. 55049. 60124. 60128. 60728. 61952.]

Column: Time Start ETD check
[50601. 50741. 52600. 52814. 53056. 53143. 53636. 54343. 54420. 54516.]

Column: Time End ETD check
[50608. 50748. 52610. 52821. 53100. 53152. 53640. 54347. 54423. 54522.]

Column: Time Start Baggage reclaim
[50552.0 50610.0 50735.0 50758.0 51115.0 551112.0 51137.0 51208.0 51222.0
 51218.0]

Column: 

In [113]:
df["checkpoint_time"] = (
    df["Time End Baggage reclaim_sec"] - df["Time Start Baggage drop off_sec"]
)

In [114]:
df["drop_time"] = (
    df["Time End Baggage drop off_sec"] - df["Time Start Baggage drop off_sec"]
)

df["wait_I"] = (
    df["Time Start WTMD_sec"] - df["Time End Baggage drop off_sec"]
)

df["collect_time"] = (
    df["Time End Baggage reclaim_sec"] - df["Time Start Baggage reclaim_sec"]
)

df["patdown_time"] = (
    df["Time End WTMD check_sec"] - df["Time Start WTMD check_sec"]
)

df["etd_time"] = (
    df["Time End ETD check_sec"] - df["Time Start ETD check_sec"]
)

df["baggage_check_time"] = (
    df["Time End Baggage Check_sec"] - df["Time Start Baggage Check_sec"]
)

In [115]:
df["has_second_wtmd"] = df["Time Start WTMD 2_sec"].notna().astype(int)
df["has_third_wtmd"] = df["Time Start WTMD 3_sec"].notna().astype(int)
df["has_patdown"] = df["Time Start WTMD check_sec"].notna().astype(int)
df["has_etd"] = df["Time Start ETD check_sec"].notna().astype(int)
df["has_baggage_check"] = df["Time Start Baggage Check_sec"].notna().astype(int)

In [116]:
duration_cols = [
    "checkpoint_time",
    "drop_time",
    "wait_I",
    "collect_time",
    "patdown_time",
    "etd_time",
    "baggage_check_time"
]

df[duration_cols].describe()

,checkpoint_time,drop_time,wait_I,collect_time,patdown_time,etd_time,baggage_check_time
count,2396.000000,2404.000000,2399.000000,2393.000000,213.000000,308.000000,168.000000
mean,179.101419,78.770799,11.640684,58.568742,22.399061,12.146104,59.732143
std,71.857038,739.172508,739.747355,50.075407,18.566693,15.416099,53.685096
min,40.000000,-3515.000000,-35993.000000,0.000000,2.000000,1.000000,3.000000
25%,130.000000,41.000000,5.000000,24.000000,13.000000,4.000000,22.500000
50%,167.000000,58.000000,12.000000,49.000000,21.000000,7.000000,42.000000
75%,215.250000,82.000000,34.000000,81.000000,28.000000,13.000000,75.250000
max,804.000000,36079.000000,3614.000000,729.000000,156.000000,97.000000,276.000000


In [117]:
for col in duration_cols:
    print(col, (df[col] < 0).sum())

checkpoint_time 0
drop_time 2
wait_I 1
collect_time 0
patdown_time 0
etd_time 0
baggage_check_time 0


In [118]:
for col in duration_cols:
    df.loc[df[col] < 0, col] = np.nan

In [119]:
df = df[df["checkpoint_time"].notna()].copy()
print(df.shape)

(2396, 51)


In [120]:
df = df[(df["checkpoint_time"] > 0) & (df["checkpoint_time"] < 3600)].copy()
print(df.shape)

(2396, 51)


In [121]:
def get_passenger_type(row):
    for col in passenger_type_cols:
        if row[col] == 1:
            return col
    return "unknown"

df["passenger_type"] = df.apply(get_passenger_type, axis=1)
df["passenger_type"].value_counts()

passenger_type
regular     843
senior      520
business    443
young       308
family      226
unknown      45
PRM          11
Name: count, dtype: int64

In [122]:
df.to_csv("../data/processed/security_processed.csv", index=False)
print("Saved processed dataset.")

Saved processed dataset.


In [123]:
df[[
    "passenger_type",
    "Experience",
    "Number of boxes",
    "Group size",
    "checkpoint_time",
    "drop_time",
    "wait_I",
    "collect_time"
]].head()

,passenger_type,Experience,Number of boxes,Group size,checkpoint_time,drop_time,wait_I,collect_time
0,regular,0,1.0,2.0,204.0,78.0,9.0,100.0
1,regular,0,2.0,2.0,223.0,88.0,17.0,103.0
2,regular,0,1.0,2.0,80.0,25.0,2.0,39.0
3,regular,0,2.0,2.0,125.0,29.0,4.0,61.0
4,business,1,1.0,1.0,125.0,46.0,3.0,58.0
